In [0]:
%pip install kafka-python

In [0]:
aiven_password = dbutils.secrets.get(
    scope="ecommerce_scope",
    key="aiven_password"
)

print("Aiven password loaded successfully!")

Aiven password loaded successfully!


In [0]:
from kafka import KafkaProducer

aiven_password = dbutils.secrets.get(
    scope="ecommerce_scope",
    key="aiven_password"
)

ca_path = "/Volumes/ecommerce_catalog/order_processing/raw_files/ca.pem"

producer = KafkaProducer(
    bootstrap_servers="ecommerce-kafka-muflihamustafa-4648.a.aivencloud.com:15690",
    security_protocol="SASL_SSL",
    sasl_mechanism="SCRAM-SHA-256",
    sasl_plain_username="avnadmin",
    sasl_plain_password=aiven_password,
    ssl_cafile=ca_path
)

print("Aiven Kafka connection successful!")

Aiven Kafka connection successful!


In [0]:
from kafka import KafkaAdminClient

admin = KafkaAdminClient(
    bootstrap_servers="ecommerce-kafka-muflihamustafa-4648.a.aivencloud.com:15690",
    security_protocol="SASL_SSL",
    sasl_mechanism="SCRAM-SHA-256",
    sasl_plain_username="avnadmin",
    sasl_plain_password=aiven_password,
    ssl_cafile=ca_path
)

topics = admin.list_topics()

print("Available Kafka topics:")
for topic in topics:
    print(topic)

Available Kafka topics:
ecommerce_orders


In [0]:
from kafka import KafkaProducer
import json

producer = KafkaProducer(
    bootstrap_servers="ecommerce-kafka-muflihamustafa-4648.a.aivencloud.com:15690",
    security_protocol="SASL_SSL",
    sasl_mechanism="SCRAM-SHA-256",
    sasl_plain_username="avnadmin",
    sasl_plain_password=aiven_password,
    ssl_cafile=ca_path,
    value_serializer=lambda x: json.dumps(x).encode("utf-8")
)

print("JSON Kafka producer ready!")

JSON Kafka producer ready!


In [0]:
future = producer.send("ecommerce_orders", value=test_order)

metadata = future.get(timeout=30)

print("Test order sent successfully!")
print("Topic:", metadata.topic)
print("Partition:", metadata.partition)
print("Offset:", metadata.offset)

Test order sent successfully!
Topic: ecommerce_orders
Partition: 0
Offset: 1462


In [0]:
from kafka import KafkaConsumer, TopicPartition
import json

consumer = KafkaConsumer(
    bootstrap_servers="ecommerce-kafka-muflihamustafa-4648.a.aivencloud.com:15690",
    security_protocol="SASL_SSL",
    sasl_mechanism="SCRAM-SHA-256",
    sasl_plain_username="avnadmin",
    sasl_plain_password=aiven_password,
    ssl_cafile=ca_path,
    value_deserializer=lambda x: json.loads(x.decode("utf-8")),
    enable_auto_commit=False
)

tp = TopicPartition("ecommerce_orders", 0)
consumer.assign([tp])
consumer.seek(tp, 1462)

records = consumer.poll(timeout_ms=10000, max_records=1)

for messages in records.values():
    for message in messages:
        print("Received order:", message.value)

consumer.close()

/home/spark-d3281720-a969-4342-83f7-6e/.ipykernel/67/command-6553420593845939-2197404338:4: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


Received order: {'order_id': 'TEST_CONNECTION_001', 'customer_id': 'C999', 'product': 'Laptop', 'quantity': 1, 'unit_price': 45000, 'total_amount': 45000, 'order_date': '2026-11-10', 'status': 'TEST'}


In [0]:
orders_df = spark.table(
    "ecommerce_catalog.order_processing.streaming_orders"
)

display(orders_df.limit(10))

order_id,customer_id,product,quantity,unit_price,order_timestamp,status
ORD1279,C190,Keyboard,1,1200.0,2026-11-05 13:43:14,PLACED
ORD1280,C185,Mobile,2,20000.0,2026-11-05 13:43:19,PLACED
ORD1281,C150,Mobile,4,20000.0,2026-11-05 13:43:25,PLACED
ORD1282,C132,Mouse,5,500.0,2026-11-05 13:43:30,PLACED
ORD1283,C174,Mobile,3,20000.0,2026-11-05 13:43:35,PLACED
ORD1284,C157,Headphones,4,2500.0,2026-11-05 13:43:40,PLACED
ORD1285,C179,Mouse,2,500.0,2026-11-05 13:43:45,PLACED
ORD1286,C136,Laptop,2,45000.0,2026-11-05 13:43:50,PLACED
ORD1287,C146,Keyboard,5,1200.0,2026-11-05 13:43:55,PLACED
ORD1288,C105,Keyboard,2,1200.0,2026-11-05 13:44:00,PLACED


In [0]:
from pyspark.sql.functions import count, countDistinct

orders_df = spark.table(
    "ecommerce_catalog.order_processing.streaming_orders"
)

orders_df.select(
    count("*").alias("total_records"),
    countDistinct("order_id").alias("unique_orders")
).display()

total_records,unique_orders
93,92


In [0]:
# Get Aiven password from Databricks secrets
aiven_password = dbutils.secrets.get(
    scope="ecommerce_scope",
    key="aiven_password"
)

# Read CA certificate
ca_pem = dbutils.fs.head(
    "/Volumes/ecommerce_catalog/order_processing/raw_files/ca.pem",
    10000
)

# Kafka settings
stream_options = {
    "kafka.bootstrap.servers": "ecommerce-kafka-muflihamustafa-4648.a.aivencloud.com:15690",
    "subscribe": "ecommerce_orders",
    "startingOffsets": "latest",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "SCRAM-SHA-256",
    "kafka.sasl.jaas.config": (
        'kafkashaded.org.apache.kafka.common.security.scram.ScramLoginModule '
        'required username="avnadmin" '
        f'password="{aiven_password}";'
    ),
    "kafka.ssl.truststore.type": "PEM",
    "kafka.ssl.truststore.certificates": ca_pem
}

kafka_stream = (
    spark.readStream
    .format("kafka")
    .options(**stream_options)
    .load()
)

streaming_orders = kafka_stream.selectExpr(
    "CAST(value AS STRING) AS order_message",
    "timestamp AS kafka_timestamp",
    "topic",
    "partition",
    "offset"
)

streaming_query = (
    streaming_orders.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option(
        "checkpointLocation",
        "/Volumes/ecommerce_catalog/order_processing/raw_files/streaming_orders_checkpoint"
    )
    .toTable(
        "ecommerce_catalog.order_processing.streaming_orders_raw"
    )
)

streaming_query.awaitTermination()

print("New Kafka messages saved!")

New Kafka messages saved!


In [0]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import *

raw_df = spark.table(
    "ecommerce_catalog.order_processing.streaming_orders_raw"
)

print("Raw Kafka data loaded!")

Raw Kafka data loaded!


In [0]:
order_schema = StructType([
    StructField("order_id", StringType()),
    StructField("customer_id", StringType()),
    StructField("product", StringType()),
    StructField("quantity", IntegerType()),
    StructField("unit_price", DoubleType()),
    StructField("order_timestamp", StringType()),
    StructField("status", StringType())
])

parsed_orders = raw_df.select(
    from_json(col("order_message"), order_schema).alias("order")
).select("order.*")

display(parsed_orders)

order_id,customer_id,product,quantity,unit_price,order_timestamp,status
ORD1280,C185,Mobile,2,20000.0,2026-11-05 13:43:19,PLACED
ORD1281,C150,Mobile,4,20000.0,2026-11-05 13:43:25,PLACED
ORD1282,C132,Mouse,5,500.0,2026-11-05 13:43:30,PLACED
ORD1283,C174,Mobile,3,20000.0,2026-11-05 13:43:35,PLACED
ORD1284,C157,Headphones,4,2500.0,2026-11-05 13:43:40,PLACED
ORD1285,C179,Mouse,2,500.0,2026-11-05 13:43:45,PLACED
ORD1286,C136,Laptop,2,45000.0,2026-11-05 13:43:50,PLACED
ORD1287,C146,Keyboard,5,1200.0,2026-11-05 13:43:55,PLACED
ORD1288,C105,Keyboard,2,1200.0,2026-11-05 13:44:00,PLACED
ORD1289,C143,Laptop,3,45000.0,2026-11-05 13:44:05,PLACED


In [0]:
from pyspark.sql.functions import count, col

parsed_orders.select(
    count("*").alias("total_orders"),
    count(col("order_id")).alias("valid_order_ids"),
    count(col("order_timestamp")).alias("valid_timestamps"),
    count(col("quantity")).alias("valid_quantities"),
    count(col("unit_price")).alias("valid_prices")
).display()

display(parsed_orders.groupBy("status").count())

total_orders,valid_order_ids,valid_timestamps,valid_quantities,valid_prices
92,92,92,92,92


status,count
PLACED,92


In [0]:
from pyspark.sql.functions import countDistinct, count

parsed_orders.select(
    count("*").alias("total_records"),
    countDistinct("order_id").alias("unique_orders")
).display()

total_records,unique_orders
92,92


In [0]:
clean_orders = (
    parsed_orders
    .filter(col("status") == "PLACED")
    .withColumn(
        "total_amount",
        col("quantity") * col("unit_price")
    )
    .withColumn(
        "order_date",
        col("order_timestamp").cast("timestamp").cast("date")
    )
)

display(clean_orders)

order_id,customer_id,product,quantity,unit_price,order_timestamp,status,total_amount,order_date
ORD1280,C185,Mobile,2,20000.0,2026-11-05 13:43:19,PLACED,40000.0,2026-11-05
ORD1281,C150,Mobile,4,20000.0,2026-11-05 13:43:25,PLACED,80000.0,2026-11-05
ORD1282,C132,Mouse,5,500.0,2026-11-05 13:43:30,PLACED,2500.0,2026-11-05
ORD1283,C174,Mobile,3,20000.0,2026-11-05 13:43:35,PLACED,60000.0,2026-11-05
ORD1284,C157,Headphones,4,2500.0,2026-11-05 13:43:40,PLACED,10000.0,2026-11-05
ORD1285,C179,Mouse,2,500.0,2026-11-05 13:43:45,PLACED,1000.0,2026-11-05
ORD1286,C136,Laptop,2,45000.0,2026-11-05 13:43:50,PLACED,90000.0,2026-11-05
ORD1287,C146,Keyboard,5,1200.0,2026-11-05 13:43:55,PLACED,6000.0,2026-11-05
ORD1288,C105,Keyboard,2,1200.0,2026-11-05 13:44:00,PLACED,2400.0,2026-11-05
ORD1289,C143,Laptop,3,45000.0,2026-11-05 13:44:05,PLACED,135000.0,2026-11-05


In [0]:
spark.table(
    "ecommerce_catalog.order_processing.streaming_orders"
).printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- order_timestamp: string (nullable = true)
 |-- status: string (nullable = true)



In [0]:
clean_orders.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "ecommerce_catalog.order_processing.processed_orders"
    )

print("Processed orders table created successfully!")

Processed orders table created successfully!


In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "ecommerce_catalog.order_processing.processed_orders"
)

(
    target.alias("target")
    .merge(
        clean_orders.alias("source"),
        "target.order_id = source.order_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Orders merged successfully!")

Orders merged successfully!


In [0]:
final_df = spark.table(
    "ecommerce_catalog.order_processing.processed_orders"
)

print("Total processed orders:", final_df.count())

display(final_df.orderBy("order_timestamp", ascending=False))

Total processed orders: 92


order_id,customer_id,product,quantity,unit_price,order_timestamp,status,total_amount,order_date
ORD1370,C199,Keyboard,3,1200.0,2026-11-05 14:09:41,PLACED,3600.0,2026-11-05
ORD1369,C145,Keyboard,3,1200.0,2026-11-05 14:09:36,PLACED,3600.0,2026-11-05
ORD1368,C185,Keyboard,3,1200.0,2026-11-05 14:09:31,PLACED,3600.0,2026-11-05
ORD1367,C109,Keyboard,2,1200.0,2026-11-05 14:09:26,PLACED,2400.0,2026-11-05
ORD1366,C135,Mouse,5,500.0,2026-11-05 14:09:21,PLACED,2500.0,2026-11-05
ORD1365,C133,Keyboard,3,1200.0,2026-11-05 14:09:16,PLACED,3600.0,2026-11-05
ORD1364,C146,Keyboard,5,1200.0,2026-11-05 14:09:11,PLACED,6000.0,2026-11-05
ORD1363,C163,Headphones,5,2500.0,2026-11-05 14:09:06,PLACED,12500.0,2026-11-05
ORD1362,C151,Mobile,2,20000.0,2026-11-05 14:09:01,PLACED,40000.0,2026-11-05
ORD1361,C184,Keyboard,1,1200.0,2026-11-05 14:08:56,PLACED,1200.0,2026-11-05


In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "ecommerce_catalog.order_processing.processed_orders"
)

(
    target.alias("target")
    .merge(
        clean_orders.alias("source"),
        "target.order_id = source.order_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Orders merged successfully!")

Orders merged successfully!
